In [2]:
%pip install great_expectations==1.24.1

In [3]:
import great_expectations as gx

print(gx.__version__)

1.24.1


In [4]:
import pandas as pd
import numpy as np
import great_expectations as gx

In [5]:
df = pd.read_csv("cleaned_dataset.csv")

print("Dataset Shape:", df.shape)

display(df.head())

Dataset Shape: (10, 6)


,Employee_ID,Name,Department,Age,Salary,Experience
0,101,Asha,It,25.0,50000.0,2.0
1,102,Rahul,Hr,30.0,45000.0,5.0
2,103,Priya,Sales,38.5,60000.0,3.0
3,104,Kiran,It,28.0,50000.0,4.0
4,105,Sneha,Sales,38.5,55000.0,6.0


## Six DAMA Data Quality Dimensions

The six dimensions used to assess data quality are:

1. Completeness
2. Accuracy
3. Consistency
4. Timeliness
5. Uniqueness
6. Validity

In [6]:
quality_assessment = pd.DataFrame({
    'Dimension': [
        'Completeness',
        'Accuracy',
        'Consistency',
        'Timeliness',
        'Uniqueness',
        'Validity'
    ],
    'Assessment': [
        'Check missing values',
        'Check age and salary ranges',
        'Check department naming consistency',
        'Requires ingestion timestamp and SLA',
        'Check duplicate employee IDs',
        'Check valid data formats and values'
    ],
    'Status': [
        'Passed' if df.isnull().sum().sum() == 0 else 'Needs Attention',
        'Passed' if df['Age'].between(18, 100).all() else 'Needs Attention',
        'Passed' if df['Department'].str.strip().str.title().equals(df['Department']) else 'Needs Attention',
        'Not Measurable',
        'Passed' if df['Employee_ID'].is_unique else 'Needs Attention',
        'Passed' if df['Salary'].ge(0).all() else 'Needs Attention'
    ]
})

display(quality_assessment)

,Dimension,Assessment,Status
0,Completeness,Check missing values,Passed
1,Accuracy,Check age and salary ranges,Passed
2,Consistency,Check department naming consistency,Passed
3,Timeliness,Requires ingestion timestamp and SLA,Not Measurable
4,Uniqueness,Check duplicate employee IDs,Passed
5,Validity,Check valid data formats and values,Passed


In [7]:
print("Total Records:", len(df))

print("Missing Values:", df.isnull().sum().sum())

print("Duplicate Records:", df.duplicated().sum())

print("Unique Employee IDs:", df['Employee_ID'].nunique())

print("Invalid Ages:", (~df['Age'].between(18, 100)).sum())

print("Negative Salaries:", (df['Salary'] < 0).sum())

Total Records: 10
Missing Values: 0
Duplicate Records: 0
Unique Employee IDs: 10
Invalid Ages: 0
Negative Salaries: 0


In [8]:
context = gx.get_context()

data_source = context.data_sources.add_pandas(
    name="employee_data_source"
)

data_asset = data_source.add_dataframe_asset(
    name="employee_data"
)

batch_definition = data_asset.add_batch_definition_whole_dataframe(
    name="employee_batch"
)

batch = batch_definition.get_batch(
    batch_parameters={"dataframe": df}
)

print("Great Expectations setup completed!")

Great Expectations setup completed!


In [9]:
expectations = [

    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="Employee_ID"
    ),

    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="Name"
    ),

    gx.expectations.ExpectColumnValuesToNotBeNull(
        column="Department"
    ),

    gx.expectations.ExpectColumnValuesToBeBetween(
        column="Age",
        min_value=18,
        max_value=100
    ),

    gx.expectations.ExpectColumnValuesToBeBetween(
        column="Salary",
        min_value=0
    ),

    gx.expectations.ExpectColumnValuesToBeBetween(
        column="Experience",
        min_value=0,
        max_value=50
    ),

    gx.expectations.ExpectColumnValuesToBeUnique(
        column="Employee_ID"
    ),

    gx.expectations.ExpectColumnValuesToBeInSet(
        column="Department",
        value_set=["IT", "Hr", "Sales"]
    )
]

print("Total Expectations:", len(expectations))

Total Expectations: 8


In [10]:
results = []

for i, expectation in enumerate(expectations, start=1):

    result = batch.validate(expectation)

    results.append({
        "Expectation": i,
        "Description": str(expectation.type),
        "Status": "Passed" if result["success"] else "Failed"
    })

validation_report = pd.DataFrame(results)

display(validation_report)

print("All Expectations Passed:",
      validation_report["Status"].eq("Passed").all())

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

AttributeError: 'ExpectColumnValuesToNotBeNull' object has no attribute 'type'

In [11]:
bad_df = df.copy()

# Error 1: Introduce a duplicate Employee ID
bad_df.loc[1, 'Employee_ID'] = bad_df.loc[0, 'Employee_ID']

# Error 2: Introduce an invalid age
bad_df.loc[2, 'Age'] = 150

# Error 3: Introduce an invalid department
bad_df.loc[3, 'Department'] = 'Unknown'

display(bad_df.head())

,Employee_ID,Name,Department,Age,Salary,Experience
0,101,Asha,It,25.0,50000.0,2.0
1,101,Rahul,Hr,30.0,45000.0,5.0
2,103,Priya,Sales,150.0,60000.0,3.0
3,104,Kiran,Unknown,28.0,50000.0,4.0
4,105,Sneha,Sales,38.5,55000.0,6.0


In [12]:
bad_batch = batch_definition.get_batch(
    batch_parameters={"dataframe": bad_df}
)

bad_results = []

for i, expectation in enumerate(expectations, start=1):

    result = bad_batch.validate(expectation)

    bad_results.append({
        "Expectation": i,
        "Status": "Passed" if result["success"] else "Failed"
    })

bad_report = pd.DataFrame(bad_results)

display(bad_report)

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

,Expectation,Status
0,1,Passed
1,2,Passed
2,3,Passed
3,4,Failed
4,5,Passed
5,6,Passed
6,7,Failed
7,8,Failed


In [13]:
failed_checks = bad_report[
    bad_report['Status'] == 'Failed'
]

print("Failed Expectations:", len(failed_checks))

print("All Three Errors Detected:",
      len(failed_checks) >= 3)

Failed Expectations: 3
All Three Errors Detected: True


In [14]:
comparison = pd.DataFrame({
    'Metric': [
        'Total Records',
        'Duplicate Employee IDs',
        'Invalid Ages',
        'Invalid Departments'
    ],

    'Original Dataset': [
        len(df),
        df['Employee_ID'].duplicated().sum(),
        (~df['Age'].between(18, 100)).sum(),
        (~df['Department'].isin(['IT', 'Hr', 'Sales'])).sum()
    ],

    'Dataset With Errors': [
        len(bad_df),
        bad_df['Employee_ID'].duplicated().sum(),
        (~bad_df['Age'].between(18, 100)).sum(),
        (~bad_df['Department'].isin(['IT', 'Hr', 'Sales'])).sum()
    ]
})

display(comparison)

,Metric,Original Dataset,Dataset With Errors
0,Total Records,10,10
1,Duplicate Employee IDs,0,1
2,Invalid Ages,0,1
3,Invalid Departments,4,4


## Conclusion

In this exercise, data governance techniques were applied to an employee dataset.

The six DAMA data quality dimensions were assessed.

Great Expectations was used to automate eight data quality checks.

Three intentional data quality errors were introduced:
- Duplicate employee ID
- Invalid age
- Invalid department

The automated validation process successfully identified these errors.

A data contract was designed to define data requirements between the producer and consumer.

Finally, PII information was classified and appropriate data protection strategies were proposed.

Data governance helps maintain reliable, accurate, consistent, and secure data for business decision-making.